# 🧠 Materi 2 — Cara Kerja AI Memahami Teks
### Training NLP 

**Kunci utamanya:** komputer **tidak mengerti kata** — komputer hanya mengerti **ANGKA**.

Maka semua sistem NLP (termasuk ChatGPT/Claude) bekerja dengan alur:

```
Teks mentah → (1) PREPROCESSING → (2) UBAH MENJADI ANGKA → (3) MODEL AI
              "dibersihkan"        "direpresentasikan"       "belajar pola & memprediksi"
```

| Bagian | Yang Dipraktikkan |
|--------|-------------------|
| 2.1 | Case folding & tokenisasi |
| 2.2 | Stopword removal (library **Sastrawi** — Bahasa Indonesia) |
| 2.3 | Stemming ke kata dasar (Sastrawi) |
| 2.4 | Membuat **fungsi preprocessing lengkap** yang bisa dipakai ulang |
| 2.5 | Bag of Words: kalimat → tabel angka |
| 2.6 | TF-IDF: memberi bobot kata penting |
| 2.7 | Menghitung **kemiripan antar kalimat** (cosine similarity) |
| 2.8 | Intuisi **word embedding**: peta makna kata |
| 2.9 | **BERT & Transformer**: embedding kontekstual, semantic search, NER |

In [ ]:
# ===== Persiapan =====
# Sastrawi = library NLP khusus Bahasa Indonesia (stopword & stemming)
!pip install Sastrawi -q

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Library siap digunakan ✅")

## 2.1 Case Folding & Tokenisasi

- **Case folding** = menyeragamkan huruf menjadi huruf kecil (agar `Servis` = `servis` = `SERVIS` dianggap kata yang sama)
- **Tokenisasi** = memecah kalimat menjadi potongan kata (*token*)

In [ ]:
kalimat = "Pelayanan Bengkel AHASS sangat MEMUASKAN dan cepat!"

# Langkah 1: case folding (semua jadi huruf kecil)
kalimat_kecil = kalimat.lower()
print("Case folding :", kalimat_kecil)

# Langkah 2: tokenisasi (ambil hanya rangkaian huruf, buang tanda baca)
token = re.findall(r"[a-z]+", kalimat_kecil)
print("Tokenisasi   :", token)
print("Jumlah token :", len(token))

## 2.2 Stopword Removal — Membuang Kata Umum

**Stopword** = kata yang sangat sering muncul tetapi tidak membawa makna penting: *dan, yang, di, ke, sangat, adalah, ...*

Membuangnya membuat analisis fokus pada kata yang benar-benar bermakna.

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

daftar_stopword = set(StopWordRemoverFactory().get_stop_words())
print("Jumlah stopword Bahasa Indonesia di Sastrawi:", len(daftar_stopword))
print("Contoh 15 di antaranya:", sorted(daftar_stopword)[:15])

# Buang stopword dari token kita
token_bersih = [t for t in token if t not in daftar_stopword]
print()
print("Sebelum :", token)
print("Sesudah :", token_bersih)

## 2.3 Stemming — Mengubah Kata ke Bentuk Dasar

**Stemming** = memotong imbuhan sehingga kata kembali ke **kata dasar**:
*pelayanan → layan*, *memuaskan → muas*, *penggantian → ganti*.

Tujuannya: agar komputer tahu bahwa *melayani*, *pelayanan*, dan *dilayani* adalah **konsep yang sama**. Bahasa Indonesia kaya imbuhan (me-, di-, pe-an, ke-an, ...), sehingga tahap ini sangat berpengaruh.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stemmer = StemmerFactory().create_stemmer()

# Contoh kata-kata berimbuhan yang sering muncul di dokumen kerja
for kata in ["pelayanan", "memuaskan", "perbaikan", "menjalankan",
             "penggantian", "pengiriman", "keterlambatan"]:
    print(f"{kata:16s} → {stemmer.stem(kata)}")

In [ ]:
# Terapkan ke token hasil langkah sebelumnya
token_dasar = [stemmer.stem(t) for t in token_bersih]

print("Sebelum stemming :", token_bersih)
print("Sesudah stemming :", token_dasar)

## 2.4 Fungsi Preprocessing Lengkap (Siap Pakai Ulang)

Di dunia kerja, keempat tahap di atas digabung menjadi **satu fungsi** yang dipanggil berulang kali:

In [ ]:
def preprocess(teks):
    """Pipeline lengkap: case folding → tokenisasi → stopword → stemming.
    Mengembalikan teks bersih siap diolah model."""
    teks = teks.lower()                                     # 1. case folding
    token = re.findall(r"[a-z]+", teks)                     # 2. tokenisasi
    token = [t for t in token if t not in daftar_stopword]  # 3. stopword removal
    token = [stemmer.stem(t) for t in token]                # 4. stemming
    return " ".join(token)

# Uji dengan beberapa kalimat berbeda
for k in [
    "Pelayanan Bengkel AHASS sangat MEMUASKAN dan cepat!",
    "Penggantian oli dilakukan oleh mekanik yang berpengalaman.",
    "Keterlambatan pengiriman sparepart mengecewakan pelanggan.",
]:
    print(f"ASLI  : {k}")
    print(f"BERSIH: {preprocess(k)}")
    print()

📌 Perhatikan bagaimana kalimat yang panjang "menyusut" menjadi inti maknanya. Teks sudah bersih → saatnya **mengubahnya menjadi angka**.

## 2.5 Bag of Words — Mengubah Kalimat Menjadi Angka

**Bag of Words (BoW)** = cara paling sederhana: hitung **berapa kali** setiap kata muncul di setiap kalimat. Hasilnya tabel angka (matriks) — inilah "bahasa" yang dimengerti komputer.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

korpus = [
    "servis motor cepat dan bagus",
    "servis motor lama sekali",
    "sparepart motor asli dan bagus",
]

cv = CountVectorizer()
bow = cv.fit_transform(korpus)

# Baris = kalimat, kolom = kata, isi = jumlah kemunculan
pd.DataFrame(bow.toarray(),
             columns=cv.get_feature_names_out(),
             index=[f"Kalimat {i+1}" for i in range(len(korpus))])

**Cara membaca:** Kalimat 1 mengandung *servis* 1×, *motor* 1×, *cepat* 1×, dst. Setiap kalimat kini berbentuk **deretan angka (vektor)**.

**Manfaat preprocessing terlihat di sini:** mari bandingkan ukuran "kamus" (jumlah kolom) sebelum vs sesudah preprocessing pada kalimat yang lebih beragam:

In [ ]:
korpus_mentah = [
    "Pelayanan bengkel sangat memuaskan!",
    "Bengkel ini melayani dengan cepat.",
    "Kami puas dengan layanan mekaniknya.",
]

# Tanpa preprocessing
cv1 = CountVectorizer()
cv1.fit(korpus_mentah)

# Dengan preprocessing (fungsi kita di bagian 2.4)
korpus_bersih = [preprocess(k) for k in korpus_mentah]
cv2 = CountVectorizer()
cv2.fit(korpus_bersih)

print("Tanpa preprocessing :", len(cv1.get_feature_names_out()), "kata unik →", list(cv1.get_feature_names_out()))
print()
print("Dengan preprocessing:", len(cv2.get_feature_names_out()), "kata unik →", list(cv2.get_feature_names_out()))

✅ Preprocessing menyatukan *pelayanan / melayani / layanan* menjadi **satu konsep** (`layan`) — kamus lebih kecil, makna lebih terkumpul, model lebih mudah belajar.

## 2.6 TF-IDF — Memberi Bobot pada Kata Penting

Kelemahan BoW: semua kata dianggap sama penting. **TF-IDF** memberi bobot lebih tinggi pada kata yang:
- **sering muncul di satu dokumen** (*Term Frequency* tinggi), tetapi
- **jarang muncul di dokumen lain** (*Inverse Document Frequency* tinggi)

Intinya: *kata yang khas = kata yang penting*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
matriks = tfidf.fit_transform(korpus)

pd.DataFrame(matriks.toarray().round(2),
             columns=tfidf.get_feature_names_out(),
             index=[f"Kalimat {i+1}" for i in range(len(korpus))])

**Perhatikan:** kata **motor** (muncul di semua kalimat) bobotnya *rendah*, sedangkan **lama / sekali** (hanya di Kalimat 2) bobotnya *tinggi* — persis intuisi manusia tentang kata yang "khas".

## 2.7 Cosine Similarity — Mengukur Kemiripan Antar Kalimat

Karena kalimat sudah menjadi vektor angka, kita bisa **menghitung seberapa mirip** dua kalimat. **Cosine similarity** menghasilkan skor **0 – 1**: semakin mendekati 1 → semakin mirip.

Inilah fondasi mesin pencari yang akan kita bangun di **Materi 4**.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

kalimat_uji = [
    "servis motor cepat dan bagus",       # K1
    "servis kendaraan cepat sekali",      # K2 — mirip K1
    "harga sparepart naik tahun ini",     # K3 — topik berbeda
]

v = TfidfVectorizer()
m = v.fit_transform(kalimat_uji)

kemiripan = cosine_similarity(m).round(2)
pd.DataFrame(kemiripan, index=["K1", "K2", "K3"], columns=["K1", "K2", "K3"])

**Cara membaca:** diagonal selalu 1.00 (kalimat identik dengan dirinya sendiri). K1–K2 memiliki skor > 0 (berbagi kata *servis* dan *cepat*), sedangkan K3 skornya 0 terhadap keduanya (topik berbeda).

## 2.8 Intuisi Word Embedding — Peta Makna Kata

BoW/TF-IDF hanya mencocokkan **kata yang sama persis**. Model AI modern melangkah lebih jauh dengan **word embedding**: setiap kata menjadi titik pada "peta makna" berdimensi ratusan — dan **kata bermakna mirip letaknya berdekatan**, walau ejaannya berbeda.

Kita gambarkan intuisinya dalam peta 2 dimensi (posisi hanyalah ilustrasi konsep):

In [ ]:
# Peta ilustrasi: koordinat dibuat manual untuk menggambarkan konsep
peta = {
    # klaster kendaraan (berdekatan satu sama lain)
    "motor": (2.0, 3.0), "sepeda motor": (2.4, 3.2), "kendaraan": (1.6, 2.7),
    # klaster pelumas
    "oli": (7.0, 6.5), "pelumas": (7.4, 6.2),
    # klaster dokumen keuangan
    "invoice": (6.5, 1.5), "tagihan": (6.9, 1.8), "faktur": (6.2, 1.2),
}

plt.figure(figsize=(7, 5))
warna_klaster = ["#990011"]*3 + ["#2F3C7E"]*2 + ["#2C8A4B"]*3
for (kata, (x, y)), w in zip(peta.items(), warna_klaster):
    plt.scatter(x, y, s=140, color=w)
    plt.annotate(kata, (x, y), textcoords="offset points", xytext=(8, 5), fontsize=11)
plt.title("Ilustrasi Word Embedding: Kata Bermakna Mirip Saling Berdekatan")
plt.xlabel("dimensi 1 (ilustrasi)"); plt.ylabel("dimensi 2 (ilustrasi)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

💡 Pada model sungguhan (Word2Vec, BERT, LLM), "peta" ini dipelajari otomatis dari miliaran kalimat dan berdimensi 300–4000. Berkat embedding, komputer tahu bahwa *"motor"* dan *"sepeda motor"* bermakna sama — sesuatu yang mustahil bagi BoW/TF-IDF.

## 2.9 BERT & Transformer Embedding — Pemahaman Kontekstual

BoW/TF-IDF dan Word2Vec memiliki keterbatasan: mereka **tidak memahami konteks**. Kata "bank" dalam "bank Mandiri" dan "bank sungai" dianggap sama.

**BERT (Bidirectional Encoder Representations from Transformers)** menyelesaikan masalah ini dengan:
- Membaca kalimat **dari kedua arah** (kiri→kanan DAN kanan→kiri)
- Menghasilkan **vektor kontekstual**: makna kata bergantung pada kata di sekitarnya
- Ukuran vektor: **768 dimensi** (vs 300 pada Word2Vec)

In [ ]:
# ===== Install library yang diperlukan =====
!pip install transformers torch sentence-transformers -q

print("Library Transformer siap digunakan!")

### 2.9.1 Tokenisasi BERT — Berbeda dari Tokenisasi Tradisional

**Masalah tokenisasi tradisional (regex):** memecah berdasarkan spasi → satu kata = satu token.
Masalahnya: jika BERT belum pernah lihat kata "memuaskan", dia tidak bisa memahaminya.

**Solusi BERT: WordPiece Tokenization** — memecah kata menjadi **sub-kata** (potongan kata lebih kecil).
Ini memungkinkan BERT memahami kata baru dengan menggabungkan sub-kata yang sudah dikenal.

### Cara Kerja WordPiece:
```
Teks asli:     "Pelayanan Bengkel AHASS sangat memuaskan"
                     ↓
Step 1 - Case folding: "pelayanan bengkel ahass sangat memuaskan"
                     ↓
Step 2 - WordPiece:    ["pelayanan", "bengkel", "aha", "##ss", "sangat", "mem", "##uas", "##kan"]
                     ↓
Step 3 - Token IDs:    [  1034,       2853,     567,   8921,    867,     1524,  3821,   5103  ]
```

**Penjelasan:**
- **Token Teks** = potongan kata yang bisa dibaca manusia (sub-kata hasil pemecahan)
- **Token IDs** = kode numerik unik untuk setiap token → inilah yang dimasukkan ke model BERT
- Tanda `##` berarti sub-kita tersebut merupakan **lanjutan** dari kata sebelumnya
- `[CLS]` dan `[SEP]` adalah token khusus BERT (awal kalimat dan pemisah)

Analogi sederhana:
- **Token Teks** seperti **huruf** dalam mengeja kata
- **Token IDs** seperti **kode pos** — angka yang merepresentasikan setiap huruf/potongan kata

In [ ]:
from transformers import AutoTokenizer

# Load tokenizer multilingual (pasti tersedia dan stabil)
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

kalimatcontoh = "Pelayanan Bengkel AHASS sangat memuaskan dan cepat!"

# Tokenisasi BERT
encoded = tokenizer(kalimatcontoh, return_tensors="pt")

# Ambil token teks (buang special tokens [CLS] dan [SEP] untuk perbandingan)
token_teks = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
token_ids = encoded['input_ids'][0].tolist()

print("=" * 60)
print("PERBANDINGAN TOKENISASI")
print("=" * 60)
print(f"\nKalimat asli : {kalimatcontoh}")

print(f"\n{'='*60}")
print("TOKENISASI TRADISIONAL (regex) - hanya memecah spasi")
print("=" * 60)
token_regex = re.findall(r'[a-z]+', kalimatcontoh.lower())
print(f"Hasil   : {token_regex}")
print(f"Jumlah  : {len(token_regex)} token")

print(f"\n{'='*60}")
print("TOKENISASI BERT (WordPiece) - memecah jadi sub-kata")
print("=" * 60)
print(f"\nToken Teks (sub-kata) : {token_teks}")
print(f"Token IDs (angka)     : {token_ids}")
print(f"Jumlah token           : {len(token_teks)}")

print(f"\n{'='*60}")
print("PENJELASAN SETIAP TOKEN")
print("=" * 60)
print(f"{'Token Teks':<15} {'Token ID':<10} Keterangan")
print("-" * 60)
for teks, ids in zip(token_teks, token_ids):
    if teks == "[CLS]":
        ket = "= Tanda AWAL kalimat (khusus BERT)"
    elif teks == "[SEP]":
        ket = "= Tanda AKHIR kalimat (khusus BERT)"
    elif teks.startswith("##"):
        ket = "= Lanjutan dari kata sebelumnya (sub-kata)"
    else:
        ket = "= Kata utuh atau awal kata"
    print(f"{teks:<15} {ids:<10} {ket}")

### 2.9.2 BERT Embedding — Vektor Kontekstual

Setiap kata dikonversi menjadi vektor **768 dimensi** yang memperhatikan **seluruh konteks kalimat**.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np

# Load model dan tokenizer (multilingual, pasti tersedia)
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def get_bert_embedding(teks):
    """Mendapatkan vektor embedding dari BERT untuk satu kalimat."""
    with torch.no_grad():
        encoded = tokenizer(teks, return_tensors="pt", padding=True, truncation=True, max_length=128)
        output = model(**encoded)
        # Ambil [CLS] token sebagai representasi kalimat
        embedding = output.last_hidden_state[:, 0, :].squeeze().numpy()
    return embedding

# Contoh: dua kalimat dengan kata "cepat" tapi konteks berbeda
kalimat_a = "Motor ini cepat sekali melaju"
kalimat_b = "Pelayanan bengkel ini sangat cepat"

emb_a = get_bert_embedding(kalimat_a)
emb_b = get_bert_embedding(kalimat_b)

print("=" * 50)
print("BERT Embedding (vektor 768 dimensi)")
print("=" * 50)
print(f"\nKalimat A: {kalimat_a}")
print(f"  Bentuk vektor: {emb_a.shape}")
print(f"  10 nilai pertama: {emb_a[:10].round(4)}")
print(f"\nKalimat B: {kalimat_b}")
print(f"  Bentuk vektor: {emb_b.shape}")
print(f"  10 nilai pertama: {emb_b[:10].round(4)}")

### 2.9.3 Cosine Similarity: TF-IDF vs BERT

Mari kita bandingkan kemampuan **TF-IDF** dan **BERT** dalam mengukur kemiripan semantik.

**Hipotesis:** BERT akan lebih akurat karena memahami *makna* kata, bukan hanya kecocokan kata persis.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer

# Kalimat uji — ada pasangan yang bermiripan semantik tapi pakai kata berbeda
kalimat_uji = [
    "servis motor cepat dan bagus",              # K1
    "kendaraan dilayani dengan cepat dan memuaskan", # K2 — mirip K1 tapi kata berbeda
    "harga sparepart naik tahun ini",            # K3 — topik berbeda
    "penggantian oli selesai dengan kilat",      # K4 — mirip K1/K2 tapi kata sangat berbeda
]

# ===== METODE 1: TF-IDF =====
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(kalimat_uji)
sim_tfidf = cosine_similarity(tfidf_matrix).round(3)

# ===== METODE 2: BERT =====
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def get_bert_embedding(teks):
    with torch.no_grad():
        encoded = tokenizer(teks, return_tensors="pt", padding=True, truncation=True, max_length=128)
        output = model(**encoded)
        return output.last_hidden_state[:, 0, :].squeeze().numpy()

bert_embeddings = np.array([get_bert_embedding(k) for k in kalimat_uji])
sim_bert = cosine_similarity(bert_embeddings).round(3)

# ===== TAMPILKAN PERBANDINGAN =====
labels = ["K1", "K2", "K3", "K4"]
print("=" * 55)
print("Cosine Similarity — TF-IDF")
print("=" * 55)
print(pd.DataFrame(sim_tfidf, index=labels, columns=labels))

print("\n" + "=" * 55)
print("Cosine Similarity — BERT")
print("=" * 55)
print(pd.DataFrame(sim_bert, index=labels, columns=labels))

print("\n" + "=" * 55)
print("Perbandingan Skor Kemiripan")
print("=" * 55)
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        diff = sim_bert[i][j] - sim_tfidf[i][j]
        arah = "LEBIH TINGGI" if diff > 0 else "lebih rendah"
        print(f"{labels[i]}-{labels[j]}: TF-IDF={sim_tfidf[i][j]:.3f}  BERT={sim_bert[i][j]:.3f}  (BERT {arah} {abs(diff):.3f})")

**Amati perbedaannya:**
- **TF-IDF K1–K4**: skor **rendah** karena kata "oli" dan "kilat" tidak ada di K1
- **BERT K1–K4**: skor **lebih tinggi** karena BERT memahami bahwa "kilat" ≈ "cepat" dan "penggantian oli" ≈ "servis"

Inilah keunggulan **semantic understanding** — BERT memahami makna, bukan hanya cocok kata.

### 2.9.4 Sentence Transformers — Semantic Search Siap Pakai

**Sentence-BERT (SBERT)** dirancang khusus untuk menghitung kemiripan antar kalimat. Sangat cocok untuk:
- Pencarian dokumen berdasarkan makna (*semantic search*)
- Mengelompokkan dokumen serupa (*clustering*)
- Mendeteksi duplikasi

In [ ]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np

# Load model Sentence-BERT multilingual (mendukung 50+ bahasa termasuk Indonesia)
st_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Korpus dokumen bengkel
dokumen = [
    "Pelayanan bengkel AHASS sangat memuaskan dan cepat",
    "Mekanik berpengalaman menangani perawatan motor dengan baik",
    "Harga oli dan sparepart mengalami kenaikan signifikan",
    "Pelanggan puas dengan kecepatan servis di bengkel resmi",
    "Invoice penggantian sparepart sudah dikirim ke pelanggan",
]

# Buat embedding untuk semua dokumen
embeddings = st_model.encode(dokumen, convert_to_tensor=True)

# Hitung cosine similarity matrix
sim_matrix = util.cos_sim(embeddings, embeddings).numpy().round(3)

print("=" * 60)
print("Dokumen dalam Korpus:")
print("=" * 60)
for i, d in enumerate(dokumen):
    print(f"  D{i+1}: {d}")

print("\n" + "=" * 60)
print("Cosine Similarity Matrix (Sentence-BERT)")
print("=" * 60)
labels = [f"D{i+1}" for i in range(len(dokumen))]
print(pd.DataFrame(sim_matrix, index=labels, columns=labels))

In [ ]:
# ===== DEMO: Semantic Search =====
kueri = "Berapa lama waktu tunggu servis motor?"

# Embed kueri
kueri_embedding = st_model.encode(kueri, convert_to_tensor=True)

# Hitung similarity dengan semua dokumen
skor = util.cos_sim(kueri_embedding, embeddings)[0].numpy()

# Urutkan dari skor tertinggi
peringkat = np.argsort(skor)[::-1]

print("=" * 60)
print(f"Kueri: {kueri}")
print("=" * 60)
print("\nHasil Pencarian (diurutkan dari paling relevan):")
print("-" * 60)
for rank, idx in enumerate(peringkat, 1):
    print(f"  Peringkat {rank}: [Skor: {skor[idx]:.3f}] {dokumen[idx]}")

**Hasil menarik:**
- Kueri "Berapa lama waktu tunggu servis motor?" **tidak memiliki kata yang sama persis** dengan dokumen manapun
- Namun BERT memahami bahwa kueri ini berkaitan dengan **"servis"** dan **"waktu"** → D4 ("Pelanggan puas dengan kecepatan servis") dan D2 ("Mekanik menangani perawatan") muncul di peringkat atas
- Inilah yang tidak bisa dilakukan oleh BoW atau TF-IDF!

### 2.9.5 spaCy untuk NER (Named Entity Recognition)

**spaCy** adalah library NLP production-ready yang menyediakan:
- Named Entity Recognition (NER): mengenali nama orang, organisasi, lokasi, dll
- Part-of-Speech (POS) tagging
- Dependency parsing

Kita gunakan untuk mengenali **entitas penting** dalam dokumen kerja.

In [ ]:
!pip install spacy -q
!python -m spacy download xx_ent_wiki_sm -q

import spacy

# Load model spaCy multilingual
nlp = spacy.load("xx_ent_wiki_sm")

# Contoh dokumen kerja
dokumen_kerja = """
PT Astra Honda Motor mengirimkan 500 unit sparepart ke bengkel AHASS di Surabaya pada tanggal 15 Januari 2025.
"""

doc = nlp(dokumen_kerja)

print("=" * 55)
print("Named Entity Recognition (NER) dengan spaCy")
print("=" * 55)
print(f"\nTeks: {dokumen_kerja}\n")
print("Entitas yang dikenali:")
print("-" * 55)
for ent in doc.ents:
    print(f"  {ent.text:30s} → {ent.label_:10s} ({spacy.explain(ent.label_)})")

### 2.9.6 Perbandingan Lengkap: Semua Metode

| Metode | Dimensi | Memahami Konteks | Kecepatan | Contoh Penggunaan |
|--------|---------|-------------------|-----------|-------------------|
| **Bag of Words** | Jumlah kata unik | ❌ | ⚡⚡⚡ | Filter spam sederhana |
| **TF-IDF** | Jumlah kata unik | ❌ | ⚡⚡⚡ | Pencarian kata kunci |
| **Word2Vec** | 300 | ❌ (per kata) | ⚡⚡ | Clustering kata |
| **BERT** | 768 | ✅ | ⚡ | Semantic search, klasifikasi |
| **Sentence-BERT** | 768 | ✅ | ⚡⚡ | Pencarian dokumen |
| **spaCy** | — | ✅ | ⚡⚡⚡ | NER, parsing grammar |

In [ ]:
# ===== RINGKASAN PERBANDINGAN DIMENSI VEKTOR =====
import matplotlib.pyplot as plt
import numpy as np

metode = ["BoW/TF-IDF\n(variable)", "Word2Vec\n(300)", "BERT\n(768)", "Sentence-BERT\n(768)"]
dimensi = [50, 300, 768, 768]  # BoW/TF-IDF bervariasi tergantung korpus
konteks = [0, 0, 1, 1]  # 0=tidak kontekstual, 1=kontekstual
warna = ["#e74c3c", "#f39c12", "#2ecc71", "#3498db"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Grafik dimensi
bars = ax1.bar(metode, dimensi, color=warna, edgecolor="black", linewidth=0.5)
for bar, dim in zip(bars, dimensi):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             str(dim), ha="center", fontsize=11, fontweight="bold")
ax1.set_ylabel("Jumlah Dimensi Vektor")
ax1.set_title("Perbandingan Dimensi Vektor")
ax1.set_ylim(0, 900)

# Grafik kemampuan konteks
bars2 = ax2.bar(metode, [1]*4, color=warna, edgecolor="black", linewidth=0.5, alpha=0.3)
bars2[2].set_alpha(1.0)
bars2[3].set_alpha(1.0)
for i, (bar, k) in enumerate(zip(bars2, konteks)):
    label = "✅ Kontekstual" if k else "❌ Tidak Kontekstual"
    ax2.text(bar.get_x() + bar.get_width()/2, 0.5, label,
             ha="center", fontsize=9, fontweight="bold",
             color="white" if k else "black")
ax2.set_ylabel("")
ax2.set_yticks([])
ax2.set_title("Kemampuan Memahami Konteks")
ax2.set_ylim(0, 1.3)

plt.suptitle("Evolusi Representasi Teks dalam NLP", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

---
# 🎯 Rangkuman & Latihan Mandiri — Materi 2

| Tahap | Teknik | Fungsi Kunci |
|-------|--------|--------------|
| Bersihkan teks | case folding, tokenisasi, stopword, stemming | `preprocess()`, Sastrawi |
| Jadikan angka | Bag of Words, TF-IDF | `CountVectorizer`, `TfidfVectorizer` |
| Ukur kemiripan | cosine similarity | `cosine_similarity()` |
| Level lanjut | word embedding | dasar LLM & semantic search |
| Modern NLP | BERT, Sentence-BERT, spaCy | kontekstual embedding, NER, semantic search |

### ✍️ Latihan Mandiri
1. Jalankan `preprocess()` pada satu kalimat dari laporan/e-mail di unit kerja Anda. Kata dasar apa saja yang tersisa?
2. Tambahkan kalimat ke-4 pada `korpus` bagian 2.5, jalankan ulang BoW dan TF-IDF — amati kolom baru yang muncul.
3. Pada bagian 2.7, tambahkan kalimat *"perbaikan motor selesai dengan cepat"* — menurut Anda skornya akan tinggi terhadap K1 atau K3? Buktikan.
4. **Tantangan:** jalankan `cosine_similarity` pada korpus **yang sudah di-`preprocess()`** — apakah skor kemiripan K1–K2 naik? Mengapa?

➡️ **Lanjut ke Materi 3:** menggunakan fondasi ini untuk mengambil poin penting dari dokumen.

---
## 📝 Jawaban Latihan Mandiri

### Jawaban 1: Preprocessing Kalimat dari Laporan Kerja

Kita jalankan `preprocess()` pada kalimat contoh dari laporan kerja di bengkel AHASS:

In [ ]:
kalimat_laporan = "Pelayanan bengkel AHASS sangat memuaskan, mekanik yang bekerja cepat dan profesional"

print("Kalimat asli :", kalimat_laporan)
print("Hasil preprocess:", preprocess(kalimat_laporan))
print()
print("Kata dasar yang tersisa:")
print([stemmer.stem(t) for t in re.findall(r"[a-z]+", kalimat_laporan.lower()) if t not in daftar_stopword])

**Penjelasan:**
- Kata "Pelayanan" → "pelayanan" (case folding) → "layan" (stemming)
- Kata "bengkel" → tetap "bengkel" (sudah kata dasar)
- Kata "AHASS" → "ahass" (case folding)
- Kata "sangat" → dibuang (stopword)
- Kata "memuaskan" → "muas" (stemming)
- Kata "mekanik" → tetap "mekanik" (sudah kata dasar)
- Kata "yang" → dibuang (stopword)
- Kata "bekerja" → "kerja" (stemming)
- Kata "cepat" → tetap "cepat" (sudah kata dasar)
- Kata "dan" → dibuang (stopword)
- Kata "profesional" → "profesional" (sudah kata dasar)

Kata dasar yang tersisa: **layan, bengkel, ahass, muas, mekanik, kerja, cepat, profesional**

### Jawaban 2: Menambah Kalimat ke-4 pada Korpus BoW & TF-IDF

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd

# Korpus asli (3 kalimat)
korpus_lama = [
    "servis motor cepat dan bagus",
    "servis motor lama sekali",
    "sparepart motor asli dan bagus",
]

# Tambahkan kalimat ke-4
korpus_baru = [
    "servis motor cepat dan bagus",
    "servis motor lama sekali",
    "sparepart motor asli dan bagus",
    "servis sparepart bagus dan murah",  # kalimat ke-4
]

# BoW - Korpus Lama
cv_lama = CountVectorizer()
bow_lama = cv_lama.fit_transform(korpus_lama)
print("=== BoW Korpus Lama (3 kalimat) ===")
print("Kata unik:", list(cv_lama.get_feature_names_out()))
print(pd.DataFrame(bow_lama.toarray(), columns=cv_lama.get_feature_names_out(),
                   index=[f"K{i+1}" for i in range(len(korpus_lama))]))

print()

# BoW - Korpus Baru
cv_baru = CountVectorizer()
bow_baru = cv_baru.fit_transform(korpus_baru)
print("=== BoW Korpus Baru (4 kalimat) ===")
print("Kata unik:", list(cv_baru.get_feature_names_out()))
print(pd.DataFrame(bow_baru.toarray(), columns=cv_baru.get_feature_names_out(),
                   index=[f"K{i+1}" for i in range(len(korpus_baru))]))

In [ ]:
# TF-IDF - Korpus Lama
tfidf_lama = TfidfVectorizer()
tfidf_matrix_lama = tfidf_lama.fit_transform(korpus_lama)
print("=== TF-IDF Korpus Lama (3 kalimat) ===")
print("Kata unik:", list(tfidf_lama.get_feature_names_out()))
print(pd.DataFrame(tfidf_matrix_lama.toarray().round(2), columns=tfidf_lama.get_feature_names_out(),
                   index=[f"K{i+1}" for i in range(len(korpus_lama))]))

print()

# TF-IDF - Korpus Baru
tfidf_baru = TfidfVectorizer()
tfidf_matrix_baru = tfidf_baru.fit_transform(korpus_baru)
print("=== TF-IDF Korpus Baru (4 kalimat) ===")
print("Kata unik:", list(tfidf_baru.get_feature_names_out()))
print(pd.DataFrame(tfidf_matrix_baru.toarray().round(2), columns=tfidf_baru.get_feature_names_out(),
                   index=[f"K{i+1}" for i in range(len(korpus_baru))]))

**Penjelasan:**
- Kata baru yang muncul: **"murah"** (dari kalimat ke-4)
- Bobot TF-IDF berubah karena distribusi kata berbeda:
  - Kata "bagus" sekarang muncul di 2 dari 4 kalimat (bobotnya berkurang)
  - Kata "murah" hanya muncul di K4 (bobotnya tinggi karena unik/khas)
  - Kata "motor" tetap rendah karena muncul di semua kalimat

### Jawaban 3: Cosine Similarity dengan Kalimat Baru

Kalimat "perbaikan motor selesai dengan cepat" memiliki kata **"motor"** dan **"cepat"** — sama dengan K1. Seharusnya skornya **lebih tinggi terhadap K1** daripada K3.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

kalimat_uji = [
    "servis motor cepat dan bagus",       # K1
    "servis kendaraan cepat sekali",      # K2 — mirip K1
    "harga sparepart naik tahun ini",     # K3 — topik berbeda
    "perbaikan motor selesai dengan cepat", # K4 — kalimat baru
]

v = TfidfVectorizer()
m = v.fit_transform(kalimat_uji)

kemiripan = cosine_similarity(m).round(2)
print("=== Cosine Similarity dengan Kalimat Baru ===")
print(pd.DataFrame(kemiripan, index=["K1", "K2", "K3", "K4"], columns=["K1", "K2", "K3", "K4"]))

print()
print("Skor K4 terhadap K1:", kemiripan[3][0])
print("Skor K4 terhadap K3:", kemiripan[3][2])
print()
if kemiripan[3][0] > kemiripan[3][2]:
    print("✅ K4 lebih mirip dengan K1 (topik servis/cepat)")
else:
    print("✅ K4 lebih mirip dengan K3 (topik berbeda)")

**Penjelasan:**
- K4 berbagi kata **"motor"** dan **"cepat"** dengan K1 → skor lebih tinggi
- K4 tidak berbagi kata dengan K3 (topik sparepart/harga) → skor rendah
- Prediksi benar: **K4 lebih mirip K1** karena memiliki kata kunci yang sama

### Jawaban 4: Cosine Similarity pada Korpus yang Sudah Di-preprocess

Pertanyaan: Apakah skor kemiripan K1–K2 **naik** setelah preprocessing? Mengapa?

**Jawaban: Ya, skor naik!** Karena preprocessing menyatukan kata berimbuhan yang bermakna sama.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Korpus ASLI (tanpa preprocessing)
kalimat_asli = [
    "servis motor cepat dan bagus",
    "servis kendaraan cepat sekali",
    "harga sparepart naik tahun ini",
]

# Korpus SUDAH di-preprocess
kalimat_bersih = [preprocess(k) for k in kalimat_asli]

print("Korpus Asli:")
for i, k in enumerate(kalimat_asli):
    print(f"  K{i+1}: {k}")

print("\nKorpus Setelah Preprocessing:")
for i, k in enumerate(kalimat_bersih):
    print(f"  K{i+1}: {k}")

# Cosine Similarity - Tanpa Preprocessing
tfidf1 = TfidfVectorizer()
m1 = tfidf1.fit_transform(kalimat_asli)
sim1 = cosine_similarity(m1).round(2)

print("\n=== Cosine Similarity TANPA Preprocessing ===")
print(pd.DataFrame(sim1, index=["K1", "K2", "K3"], columns=["K1", "K2", "K3"]))

# Cosine Similarity - Dengan Preprocessing
tfidf2 = TfidfVectorizer()
m2 = tfidf2.fit_transform(kalimat_bersih)
sim2 = cosine_similarity(m2).round(2)

print("\n=== Cosine Similarity DENGAN Preprocessing ===")
print(pd.DataFrame(sim2, index=["K1", "K2", "K3"], columns=["K1", "K2", "K3"]))

print("\n=== Perbandingan Skor K1–K2 ===")
print(f"Tanpa preprocessing : {sim1[0][1]}")
print(f"Dengan preprocessing: {sim2[0][1]}")

if sim2[0][1] > sim1[0][1]:
    print(f"\n✅ Skor NAIK dari {sim1[0][1]} menjadi {sim2[0][1]}")
    print("Mengapa? Preprocessing mengubah 'kendaraan' dan 'servis' ke bentuk dasar yang sama,")
    print("sehingga更多 kata yang cocok antara K1 dan K2.")
else:
    print(f"\nSkor tetap atau turun: {sim2[0][1]}")

**Kesimpulan Jawaban 4:**

Skor kemiripan K1–K2 **NAIK** setelah preprocessing karena:

1. **Kata "kendaraan" → "kira"** (stemming mengubah ke bentuk dasar)
2. **Kata "servis" → "servis"** (tetap sama)
3. **Kata "cepat" → "cepat"** (tetap sama)
4. Kata-kata umum seperti "dan", "sekali" dibuang (stopword removal)

Hasilnya: K1 dan K2 sekarang berbagi lebih banyak kata yang sama setelah preprocessing, sehingga **skor kemiripan meningkat**. Ini membuktikan preprocessing sangat penting untuk analisis teks!